# Condition Ablation Diagnostics

DINO context token `C_E`가 실제로 DiT 예측에 쓰이는지 확인합니다.

진단 항목:

1. `real`: 원래 조건 `C_E + C_m`
2. `motion_only`: `C_E` 제거, `C_m`만 사용
3. `ce_shuffled`: 다른 video의 DINO token을 사용하고 motion은 원본 유지
4. `background_only`: foreground token 제거, low-motion background token만 사용
5. cross-attention group map: 미래 latent token이 `C_E foreground`, `C_E background`, `C_m` 중 어디를 보는지 확인

해석 기준:

- `motion_only`, `ce_shuffled`, `background_only`가 `real`과 거의 같으면 DINO foreground condition이 약하게 쓰이는 것입니다.
- cross-attention에서 `C_m`만 크거나 `C_E foreground`가 작으면 객체 appearance token이 미래 latent 생성에 잘 연결되지 않는 것입니다.


In [ ]:
from __future__ import annotations

from contextlib import contextmanager
from dataclasses import replace
from pathlib import Path
import sys
import json
import math
import random
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.pipelines.latent_dit_pipeline import (
    FeatureLatentDataset,
    aggregate_frame_scores,
    apply_score_normalization,
    build_flow_matching_sampler,
    compute_score_metrics,
    feature_collate,
    load_feature_records,
    load_flow_matching_config,
    load_pipeline_config,
    load_torch_payload,
    make_loader_kwargs,
    per_frame_latent_scores,
)
from src.pipelines.training_pipeline import configure_torch_backend, get_device, load_checkpoint_for_inference

try:
    from sklearn.metrics import average_precision_score, roc_auc_score
except Exception:  # pragma: no cover - notebook fallback
    average_precision_score = None
    roc_auc_score = None

pd.set_option("display.max_columns", 80)
plt.rcParams["figure.dpi"] = 120


## 설정

In [ ]:
RUN_ID = "flow_dit_fg_bg_768x16_001"
CONFIG_PATH = ROOT / "config/local.yaml"
CHECKPOINT_PATH = ROOT / "checkpoints" / RUN_ID / "best.pt"
PREDICTION_DIR = ROOT / "data/04_predictions/shanghaitech" / RUN_ID
FEATURE_INDEX = ROOT / "data/03_features/shanghaitech/test_feature_index.jsonl"
OUTPUT_DIR = ROOT / "notebooks/outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Existing analysis showed this video is a hard failure in scene 12.
TARGET_VIDEO_ID = "12_0149"
WINDOWS_PER_CLASS = 12
BATCH_SIZE = 4
SEED = 42

# Keep this equal to inference config unless intentionally testing sampling quality.
INFERENCE_STEPS = 4

rng = np.random.default_rng(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

print("checkpoint", CHECKPOINT_PATH)
print("feature_index", FEATURE_INDEX)
print("prediction_dir", PREDICTION_DIR)


## 데이터 선택

In [ ]:
def evenly_take(records: list[dict[str, Any]], count: int) -> list[dict[str, Any]]:
    if len(records) <= count:
        return list(records)
    indices = np.linspace(0, len(records) - 1, count).round().astype(int)
    return [records[int(index)] for index in indices]


def future_has_anomaly(record: dict[str, Any]) -> bool:
    return any(int(value) == 1 for value in record.get("future_frame_labels", []))


all_records = load_feature_records(FEATURE_INDEX, normal_only=False, limit_samples=None)
target_records = [record for record in all_records if record["video_id"] == TARGET_VIDEO_ID]
if not target_records:
    raise ValueError(f"No records found for TARGET_VIDEO_ID={TARGET_VIDEO_ID}")

normal_records = [record for record in target_records if not future_has_anomaly(record)]
anomaly_records = [record for record in target_records if future_has_anomaly(record)]
selected_records = evenly_take(normal_records, WINDOWS_PER_CLASS) + evenly_take(
    anomaly_records, WINDOWS_PER_CLASS
)
selected_records = sorted(selected_records, key=lambda record: record["context_start_frame"])

donor_pool = [record for record in all_records if record["video_id"] != TARGET_VIDEO_ID]
if len(donor_pool) < len(selected_records):
    raise ValueError("Not enough donor records for cross-video C_E shuffle")
donor_indices = rng.choice(len(donor_pool), size=len(selected_records), replace=False)
donor_records = [donor_pool[int(index)] for index in donor_indices]

print(
    "selected windows",
    len(selected_records),
    "normal",
    len([record for record in selected_records if not future_has_anomaly(record)]),
    "anomaly/mixed",
    len([record for record in selected_records if future_has_anomaly(record)]),
)
print("target video windows", len(target_records), "normal", len(normal_records), "anomaly/mixed", len(anomaly_records))

pd.DataFrame(
    {
        "sample_id": [record["sample_id"] for record in selected_records],
        "future_start": [record["future_start_frame"] for record in selected_records],
        "future_end": [record["future_end_frame"] for record in selected_records],
        "future_label_sum": [sum(record["future_frame_labels"]) for record in selected_records],
    }
).head(30)


## 모델 로드

In [ ]:
config = load_pipeline_config(CONFIG_PATH)
configure_torch_backend(config.optimization.matmul_precision)
device = get_device()
model, latent_stats, checkpoint_config = load_checkpoint_for_inference(CHECKPOINT_PATH, device=device)
checkpoint_flow_config = load_flow_matching_config(checkpoint_config["flow_matching"])
checkpoint_flow_config = replace(checkpoint_flow_config, inference_steps=INFERENCE_STEPS)
flow = build_flow_matching_sampler(checkpoint_flow_config)
model.eval()

print("device", device)
print("flow", checkpoint_flow_config)
print("context reduction", model.context_adapter.token_reduction)
print("context max_tokens", model.context_adapter.max_tokens)
print("motion adapter", model.motion_adapter is not None)
print("params", f"{sum(parameter.numel() for parameter in model.parameters()) / 1e6:.2f}M")


## Ablation 실행

In [ ]:
def repo_path(path_like: str | Path) -> Path:
    path = Path(path_like)
    return path if path.is_absolute() else ROOT / path


def load_records_batch(records: list[dict[str, Any]]) -> dict[str, Any]:
    samples = []
    for record in records:
        context_payload = load_torch_payload(repo_path(record["context_feature_path"]))
        future_payload = load_torch_payload(repo_path(record["future_latent_path"]))
        motion = None
        if "motion_feature_path" in record:
            motion = load_torch_payload(repo_path(record["motion_feature_path"]))["motion"].float()
        samples.append(
            {
                "context": context_payload["embedding"].float(),
                "future": future_payload["latent"].float(),
                "motion": motion,
                "metadata": record,
            }
        )
    return feature_collate(samples)


def build_prediction_rows(metadata: list[dict[str, Any]], scores: torch.Tensor) -> list[dict[str, Any]]:
    rows = []
    for index, record in enumerate(metadata):
        frame_scores = [float(value) for value in scores[index].tolist()]
        rows.append(
            {
                "sample_id": record["sample_id"],
                "video_id": record["video_id"],
                "scene_id": record.get("scene_id", record.get("class_name", "unknown")),
                "future_frames": list(record["future_frames"]),
                "future_frame_labels": [int(value) for value in record["future_frame_labels"]],
                "future_frame_scores": frame_scores,
                "sample_score": float(np.mean(frame_scores)),
            }
        )
    return rows


@contextmanager
def ablate_model_context(model: nn.Module, mode: str):
    adapter = model.context_adapter
    original_forward = adapter.forward
    original_select = adapter.select_foreground_background_tokens

    if mode == "motion_only":
        def context_forward_without_tokens(context: torch.Tensor, token_scores: torch.Tensor | None = None):
            return context.new_zeros((context.shape[0], 0, adapter.hidden_size))

        adapter.forward = context_forward_without_tokens
    elif mode == "background_only":
        def select_background_only(tokens: torch.Tensor, token_scores: torch.Tensor):
            if token_scores.shape != tokens.shape[:3]:
                raise ValueError("token_scores shape mismatch")
            batch, frames, num_tokens, hidden = tokens.shape
            if adapter.max_tokens is None or frames * num_tokens <= adapter.max_tokens:
                per_frame_tokens = num_tokens
            else:
                per_frame_tokens = max(1, min(num_tokens, adapter.max_tokens // frames))
            indices = (-token_scores).topk(per_frame_tokens, dim=-1).indices
            gather_indices = indices.unsqueeze(-1).expand(batch, frames, per_frame_tokens, hidden)
            return tokens.gather(dim=2, index=gather_indices).flatten(1, 2)

        adapter.select_foreground_background_tokens = select_background_only

    try:
        yield
    finally:
        adapter.forward = original_forward
        adapter.select_foreground_background_tokens = original_select


def move_batch_to_device(batch: dict[str, Any], *, context_override: torch.Tensor | None = None):
    context = batch["context"].to(device, non_blocking=True)
    if context_override is not None:
        context = context_override.to(device, non_blocking=True)
    future = batch["future"].to(device, non_blocking=True)
    motion = batch["motion"].to(device, non_blocking=True) if batch["motion"] is not None else None
    return context, future, motion


@torch.inference_mode()
def run_ablation_mode(mode: str) -> dict[str, Any]:
    prediction_rows = []
    batch_summaries = []
    predictions_cpu = []
    futures_cpu = []

    for start in tqdm(range(0, len(selected_records), BATCH_SIZE), desc=f"mode={mode}"):
        batch_records = selected_records[start : start + BATCH_SIZE]
        batch = load_records_batch(batch_records)
        context_override = None
        if mode == "ce_shuffled":
            donor_batch = load_records_batch(donor_records[start : start + len(batch_records)])
            context_override = donor_batch["context"]

        context, future, motion = move_batch_to_device(batch, context_override=context_override)
        generator = torch.Generator(device=device).manual_seed(SEED + start)
        with ablate_model_context(model, mode):
            predicted = flow.euler_sample(
                model,
                context,
                tuple(future.shape),
                motion_features=motion,
                latent_stats=latent_stats,
                inference_steps=INFERENCE_STEPS,
                generator=generator,
            )
        scores = per_frame_latent_scores(
            future,
            predicted,
            scoring=config.scoring,
            latent_stats=latent_stats,
        )
        prediction_rows.extend(build_prediction_rows(batch["metadata"], scores.detach().cpu()))
        predictions_cpu.append(predicted.detach().cpu())
        futures_cpu.append(future.detach().cpu())
        batch_summaries.append(
            {
                "batch_start": start,
                "mse_to_future": float((predicted - future).square().mean().detach().cpu()),
                "pred_rms": float(predicted.square().mean().sqrt().detach().cpu()),
                "score_mean": float(scores.mean().detach().cpu()),
            }
        )

    frame_scores = apply_score_normalization(
        aggregate_frame_scores(prediction_rows),
        scoring=config.scoring,
    )
    metrics = compute_score_metrics(frame_scores)
    return {
        "mode": mode,
        "prediction_rows": prediction_rows,
        "frame_scores": frame_scores,
        "score_metrics": metrics,
        "batch_summaries": batch_summaries,
        "predictions": torch.cat(predictions_cpu, dim=0),
        "futures": torch.cat(futures_cpu, dim=0),
    }


modes = ["real", "motion_only", "ce_shuffled", "background_only"]
results = {mode: run_ablation_mode(mode) for mode in modes}
print("done")


## Ablation 결과 요약

In [ ]:
def extract_metric(metrics: dict[str, dict[str, Any]], score_key: str, metric_key: str) -> float:
    value = metrics.get(score_key, {}).get(metric_key)
    return float(value) if value is not None else float("nan")


real_predictions = results["real"]["predictions"]
summary_rows = []
for mode, result in results.items():
    predictions = result["predictions"]
    futures = result["futures"]
    summary_rows.append(
        {
            "mode": mode,
            "raw_auc": extract_metric(result["score_metrics"], "raw_score", "roc_auc"),
            "raw_ap": extract_metric(result["score_metrics"], "raw_score", "average_precision"),
            "video_centered_auc": extract_metric(
                result["score_metrics"], "video_centered_score", "roc_auc"
            ),
            "video_centered_ap": extract_metric(
                result["score_metrics"], "video_centered_score", "average_precision"
            ),
            "sample_mse_to_future": float((predictions - futures).square().mean()),
            "pred_rms": float(predictions.square().mean().sqrt()),
            "mse_to_real_prediction": float((predictions - real_predictions).square().mean()),
            "delta_vs_real_rms_ratio": float(
                (predictions - real_predictions).square().mean().sqrt()
                / real_predictions.square().mean().sqrt().clamp_min(1e-12)
            ),
            "frame_rows": len(result["frame_scores"]),
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

metrics_payload = {
    "run_id": RUN_ID,
    "target_video_id": TARGET_VIDEO_ID,
    "selected_sample_ids": [record["sample_id"] for record in selected_records],
    "donor_sample_ids": [record["sample_id"] for record in donor_records],
    "summary": summary_df.to_dict(orient="records"),
}
metrics_path = OUTPUT_DIR / f"condition_ablation_metrics_{RUN_ID}.json"
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
print("wrote", metrics_path)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for column, ax in zip(
    ["video_centered_auc", "sample_mse_to_future", "mse_to_real_prediction"],
    axes,
    strict=True,
):
    ax.bar(summary_df["mode"], summary_df[column])
    ax.set_title(column)
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)
fig.suptitle(f"Condition ablation on {TARGET_VIDEO_ID}")
fig.tight_layout()
ablation_plot_path = OUTPUT_DIR / f"condition_ablation_summary_{RUN_ID}.png"
fig.savefig(ablation_plot_path, bbox_inches="tight")
print("wrote", ablation_plot_path)
plt.show()


## Cross-Attention Map

In [ ]:
class CrossAttentionRecorder(nn.Module):
    def __init__(self, module: nn.MultiheadAttention, store: dict[int, torch.Tensor], layer_index: int):
        super().__init__()
        self.module = module
        self.store = store
        self.layer_index = layer_index

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, **kwargs):
        output, weights = self.module(
            query,
            key,
            value,
            key_padding_mask=kwargs.get("key_padding_mask"),
            need_weights=True,
            attn_mask=kwargs.get("attn_mask"),
            average_attn_weights=False,
            is_causal=kwargs.get("is_causal", False),
        )
        self.store[self.layer_index] = weights.detach().cpu()
        return output, weights


@contextmanager
def record_cross_attention(model: nn.Module):
    store: dict[int, torch.Tensor] = {}
    originals = []
    for layer_index, block in enumerate(model.blocks):
        originals.append((block, block.cross_attn))
        block.cross_attn = CrossAttentionRecorder(block.cross_attn, store, layer_index)
    try:
        yield store
    finally:
        for block, original in originals:
            block.cross_attn = original


def condition_group_masks(model: nn.Module, motion: torch.Tensor) -> tuple[dict[str, torch.Tensor], dict[str, int]]:
    adapter = model.context_adapter
    frames = adapter.context_frames
    num_context_tokens = adapter.context_tokens
    if adapter.max_tokens is None or frames * num_context_tokens <= adapter.max_tokens:
        per_frame_tokens = num_context_tokens
        foreground_tokens = per_frame_tokens
    else:
        per_frame_tokens = max(1, min(num_context_tokens, adapter.max_tokens // frames))
        if per_frame_tokens == 1:
            foreground_tokens = 1
        else:
            foreground_tokens = int(round(per_frame_tokens * adapter.foreground_ratio))
            foreground_tokens = min(max(foreground_tokens, 1), per_frame_tokens - 1)
    background_tokens = per_frame_tokens - foreground_tokens
    ce_tokens = frames * per_frame_tokens
    with torch.inference_mode():
        motion_tokens = model.motion_adapter(motion).shape[1] if model.motion_adapter is not None else 0
    total_tokens = ce_tokens + motion_tokens
    masks = {
        "C_E foreground": torch.zeros(total_tokens, dtype=torch.bool),
        "C_E background": torch.zeros(total_tokens, dtype=torch.bool),
        "C_m motion": torch.zeros(total_tokens, dtype=torch.bool),
    }
    for frame in range(frames):
        start = frame * per_frame_tokens
        masks["C_E foreground"][start : start + foreground_tokens] = True
        if background_tokens > 0:
            masks["C_E background"][start + foreground_tokens : start + per_frame_tokens] = True
    if motion_tokens > 0:
        masks["C_m motion"][ce_tokens:] = True
    layout = {
        "per_frame_context_tokens": per_frame_tokens,
        "foreground_tokens_per_frame": foreground_tokens,
        "background_tokens_per_frame": background_tokens,
        "ce_tokens": ce_tokens,
        "motion_tokens": motion_tokens,
        "total_condition_tokens": total_tokens,
    }
    return masks, layout


# One normal and one anomaly/mixed sample from the same hard video.
attention_records = [normal_records[0], anomaly_records[0]]
attention_batch = load_records_batch(attention_records)
context, future, motion = move_batch_to_device(attention_batch)
with torch.inference_mode():
    generator = torch.Generator(device=device).manual_seed(SEED)
    latents_at_tau, time_values, _, _ = flow.prepare_training_pair(
        future,
        latent_stats=latent_stats,
        generator=generator,
    )
    with record_cross_attention(model) as attention_store:
        _ = model(latents_at_tau, time_values, context, motion)

masks, layout = condition_group_masks(model, motion)
print(layout)
print("captured layers", sorted(attention_store.keys())[:5], "...", len(attention_store))


In [ ]:
attention_rows = []
for layer_index, weights in sorted(attention_store.items()):
    # weights: [B, heads, future_tokens, condition_tokens]
    avg_by_condition = weights.mean(dim=(0, 1, 2))
    row = {"layer": layer_index}
    for name, mask in masks.items():
        row[name] = float(avg_by_condition[mask].sum())
    attention_rows.append(row)

attention_df = pd.DataFrame(attention_rows)
display(attention_df.head())

fig, ax = plt.subplots(figsize=(9, 4))
for name in ["C_E foreground", "C_E background", "C_m motion"]:
    ax.plot(attention_df["layer"], attention_df[name], marker="o", label=name)
ax.set_title("Cross-attention share by condition group")
ax.set_xlabel("DiT layer")
ax.set_ylabel("attention share")
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
attention_plot_path = OUTPUT_DIR / f"condition_ablation_attention_groups_{RUN_ID}.png"
fig.savefig(attention_plot_path, bbox_inches="tight")
print("wrote", attention_plot_path)
plt.show()


In [ ]:
# Spatial query map for t+1 future latent patches.
# Each map shows how much each future latent patch attends to a condition group.
weights_stack = torch.stack([weights for _, weights in sorted(attention_store.items())], dim=0)
# [layers, B, heads, query, source] -> [B, query, source]
mean_attention = weights_stack.mean(dim=(0, 2))
future_adapter = model.future_adapter
num_patches = future_adapter.num_patches
grid_h = future_adapter.grid_h
grid_w = future_adapter.grid_w
first_future_queries = slice(0, num_patches)

fig, axes = plt.subplots(len(attention_records), 3, figsize=(10, 6))
for sample_index, record in enumerate(attention_records):
    for col_index, name in enumerate(["C_E foreground", "C_E background", "C_m motion"]):
        values = mean_attention[sample_index, first_future_queries, masks[name]].sum(dim=-1)
        image = values.reshape(grid_h, grid_w).numpy()
        ax = axes[sample_index, col_index]
        im = ax.imshow(image, cmap="magma")
        label_type = "normal" if not future_has_anomaly(record) else "anomaly/mixed"
        ax.set_title(f"{label_type}\n{name}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(f"t+1 future latent query attention maps: {TARGET_VIDEO_ID}")
fig.tight_layout()
attention_map_path = OUTPUT_DIR / f"condition_ablation_attention_maps_{RUN_ID}.png"
fig.savefig(attention_map_path, bbox_inches="tight")
print("wrote", attention_map_path)
plt.show()


## 자동 해석

In [ ]:
real_row = summary_df.set_index("mode").loc["real"]
interpretation_rows = []
for mode in ["motion_only", "ce_shuffled", "background_only"]:
    row = summary_df.set_index("mode").loc[mode]
    interpretation_rows.append(
        {
            "check": mode,
            "auc_delta_vs_real": float(row["video_centered_auc"] - real_row["video_centered_auc"]),
            "mse_delta_vs_real": float(row["sample_mse_to_future"] - real_row["sample_mse_to_future"]),
            "prediction_delta_ratio": float(row["delta_vs_real_rms_ratio"]),
            "flag_condition_weak": bool(
                abs(row["video_centered_auc"] - real_row["video_centered_auc"]) < 0.03
                and row["delta_vs_real_rms_ratio"] < 0.05
            ),
        }
    )
interpretation_df = pd.DataFrame(interpretation_rows)
display(interpretation_df)

attention_mean = attention_df[["C_E foreground", "C_E background", "C_m motion"]].mean()
token_counts = {
    "C_E foreground": layout["foreground_tokens_per_frame"] * model.context_adapter.context_frames,
    "C_E background": layout["background_tokens_per_frame"] * model.context_adapter.context_frames,
    "C_m motion": layout["motion_tokens"],
}
attention_summary = pd.DataFrame(
    {
        "condition_group": attention_mean.index,
        "tokens": [token_counts[name] for name in attention_mean.index],
        "uniform_expected_share": [
            token_counts[name] / layout["total_condition_tokens"] for name in attention_mean.index
        ],
        "mean_attention_share": attention_mean.values,
        "per_token_attention_share": [
            attention_mean[name] / token_counts[name] if token_counts[name] else float("nan")
            for name in attention_mean.index
        ],
    }
)
attention_summary["share_minus_uniform"] = (
    attention_summary["mean_attention_share"] - attention_summary["uniform_expected_share"]
)
display(attention_summary)

report = {
    **metrics_payload,
    "interpretation": interpretation_df.to_dict(orient="records"),
    "attention_layout": layout,
    "attention_group_mean": attention_summary.to_dict(orient="records"),
}
report_path = OUTPUT_DIR / f"condition_ablation_report_{RUN_ID}.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("wrote", report_path)
